In [1]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\91991\Desktop\internships\AMDOX\data\processed\clean_online_retail.csv"
)

print(df.shape)
df.head()


(400916, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [2]:
print("Customers:", df["Customer ID"].nunique())

print("Products:", df["StockCode"].nunique())

Customers: 4312
Products: 4017


In [3]:
# Products bought only once create noise.
product_counts = (
    df["StockCode"]
    .value_counts()
)

valid_products = product_counts[
    product_counts >= 10
].index

df = df[
    df["StockCode"].isin(valid_products)
]

print(df.shape)

(397005, 9)


In [4]:
# Create Customer-Item Matrix
customer_item_matrix = pd.pivot_table(
    df,
    index="Customer ID",
    columns="StockCode",
    values="Quantity",
    aggfunc="sum",
    fill_value=0
)

print(customer_item_matrix.shape)

(4312, 3111)


In [5]:
customer_item_matrix = (
    customer_item_matrix > 0
).astype(int)

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

item_similarity = cosine_similarity(
    customer_item_matrix.T
)
print(item_similarity.shape)

(3111, 3111)


In [7]:
similarity_df = pd.DataFrame(
    item_similarity,
    index=customer_item_matrix.columns,
    columns=customer_item_matrix.columns
)

In [8]:
similarity_df.memory_usage(
    deep=True
).sum() / 1024**2

np.float64(74.08742332458496)

In [9]:
product_map = (
    df[
        ["StockCode", "Description"]
    ]
    .drop_duplicates()
)

product_map.head()

,StockCode,Description
0,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS
1,79323P,PINK CHERRY LIGHTS
2,79323W,WHITE CHERRY LIGHTS
3,22041,"RECORD FRAME 7"" SINGLE SIZE"
4,21232,STRAWBERRY CERAMIC TRINKET BOX


In [10]:
def recommend_products(
    stock_code,
    top_n=5
):

    similar_products = (
        similarity_df[stock_code]
        .sort_values(
            ascending=False
        )
    )

    return similar_products.iloc[
        1:top_n+1
    ]

In [11]:
def recommend_products_with_names(
    stock_code,
    top_n=5
):

    recommendations = recommend_products(
        stock_code,
        top_n
    )

    results = product_map[
        product_map["StockCode"].isin(
            recommendations
        )
    ]

    return results[
        [
            "StockCode",
            "Description"
        ]
    ]

In [12]:
sample_product = similarity_df.index[0]

print("Product:", sample_product)

recommend_products_with_names(
    sample_product
)

Product: 10002


,StockCode,Description


In [22]:
recommend_products_with_names(
    "10002"
)

,StockCode,Description


In [23]:
import joblib

joblib.dump(
    similarity_df,
    r"C:\Users\91991\Desktop\internships\AMDOX\models\recommendation\item_similarity.pkl"
)

['C:\\Users\\91991\\Desktop\\internships\\AMDOX\\models\\recommendation\\item_similarity.pkl']

In [24]:
# from src.recommendation.recommender import (
#     recommend_products_with_names
# )

print(
    recommend_products_with_names(
        "85123A"
    )
)

Empty DataFrame
Columns: [StockCode, Description]
Index: []


In [25]:
import pandas as pd
import joblib

df = pd.read_csv(
    r"C:\Users\91991\Desktop\internships\AMDOX\data\processed\clean_online_retail.csv"
)

product_map = (
    df[
        ["StockCode", "Description"]
    ]
    .drop_duplicates()
)

joblib.dump(
    product_map,
    r"C:\Users\91991\Desktop\internships\AMDOX\models\recommendation\product_map.pkl"
)

print("Saved product_map.pkl")

Saved product_map.pkl
